***# This File evaluates the performance of Qwen Model after FineTuning.                                                                 To be Run after training the Model.***

In [ ]:
# Install Required packages
import subprocess, os
subprocess.run(["pip", "install", "-q", "qwen-vl-utils", "transformers==4.49.0",
                "peft", "accelerate", "bitsandbytes", "datasets",
                "pillow", "bert-score", "nltk"], check=True)


In [ ]:
import torch
torch.cuda.empty_cache()
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

import re, json, ast, random
import numpy as np
from collections import Counter
from PIL import Image
from datasets import load_dataset
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

SEED          = 42
DATASET_NAME  = "sujet-ai/Sujet-Finance-QA-Vision-100k"
CONTENT_LIMIT = 800
MAX_SEQ_LEN   = 1024
DEVICE        = "cuda"
DTYPE         = torch.float16
VAL_SAMPLES   = 500
EVAL_N        = 250       # number of samples to evaluate

# Point this to your best saved checkpoint
MODEL_BASE    = "Qwen/Qwen2-VL-2B-Instruct"
MODEL_BASE = "Qwen/Qwen2-VL-2B-Instruct"
CHECKPOINT = "/kaggle/input/datasets/bishakhasingh/qwen-adapter"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("\nLoad the processor")
processor = AutoProcessor.from_pretrained(
    MODEL_BASE,
    use_fast=False,
    min_pixels=128*28*28,
    max_pixels=256*28*28,
)

print("Load the base model")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_BASE,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print(f"Load LoRA weights from: {CHECKPOINT}")
model = PeftModel.from_pretrained(base_model, CHECKPOINT)
model.eval()

In [ ]:
def parse_qa_pairs(qa_str):
    if not qa_str: return []
    if isinstance(qa_str, list): return qa_str
    try: pairs = json.loads(qa_str)
    except Exception:
        try: pairs = ast.literal_eval(qa_str)
        except Exception: return []
    if isinstance(pairs, dict): pairs = [pairs]
    if not isinstance(pairs, list): return []
    results = []
    for p in pairs:
        if not isinstance(p, dict): continue
        q = p.get("question") or p.get("ques") or p.get("q")
        a = p.get("answer")   or p.get("ans")  or p.get("a")
        if q and a:
            results.append({"question": str(q).strip(), "answer": str(a).strip()})
    return results

def clean_content(content):
    if not content: return ""
    content = re.sub(r"#{1,3}\s*", "", content)
    content = re.sub(r"\*\*(.+?)\*\*", r"\1", content)
    content = re.sub(r"\*(.+?)\*", r"\1", content)
    content = re.sub(r"^\s*[-•]\s*", "", content, flags=re.MULTILINE)
    content = re.sub(r"\n+", " ", content)
    content = re.sub(r"\s+", " ", content)
    return content.strip()

print("\nLoad the dataset")
dataset = load_dataset(DATASET_NAME)

flat_images, flat_questions, flat_answers, flat_contents = [], [], [], []
for ex in dataset["train"]:
    content  = clean_content(ex.get("content", ""))
    qa_pairs = parse_qa_pairs(ex.get("qa_pairs"))
    for qa in qa_pairs:
        flat_images.append(ex["image"])
        flat_questions.append(qa["question"])
        flat_answers.append(qa["answer"])
        flat_contents.append(content)
        if len(flat_images) >= 5000:
            break
    if len(flat_images) >= 5000:
        break

print(f"Total QA pairs: {len(flat_images):,}")

# Reproduce same train/val split as training (same seed)
indices = list(range(len(flat_images)))
random.shuffle(indices)
n_train = int(len(indices) * 0.9)
va_idx  = indices[n_train:n_train + VAL_SAMPLES]
print(f"Val samples: {len(va_idx)}")


In [ ]:
from torch.utils.data import Dataset as TorchDataset
class FinanceVQADataset(TorchDataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        img = flat_images[i]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        return {
            "image":    img.convert("RGB"),
            "question": flat_questions[i],
            "answer":   flat_answers[i],
            "content":  flat_contents[i],
        }

val_ds = FinanceVQADataset(va_idx)
print(f"Val dataset size: {len(val_ds)}")

def build_prompt_text(item):
    return (
        "You are an expert in processing financial document.\n"
        f"Document content: {item['content'][:CONTENT_LIMIT]}\n\n"
        "Answer the question using only information from the document. "
        "Give a SHORT and DIRECT answer in maximum 1-2 sentences.\n\n"
        f"Question: {item['question']}"
    )

def run_inference(sample):
    from qwen_vl_utils import process_vision_info
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": sample["image"]},
        {"type": "text",  "text": build_prompt_text(sample)},
    ]}]
    prompt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(msgs)
    inputs = processor(text=[prompt], images=image_inputs, return_tensors="pt", padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(dtype=DTYPE)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            repetition_penalty=1.1
        )
    pred = processor.tokenizer.decode(
        out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    return pred


def normalize(s):
    s = str(s).lower().strip()
    number_map = {
        "zero":"0","one":"1","two":"2","three":"3","four":"4",
        "five":"5","six":"6","seven":"7","eight":"8","nine":"9","ten":"10"
    }
    for word, digit in number_map.items():
        s = re.sub(rf"\b{word}\b", digit, s)
    fillers = [
        "the purpose is", "the purpose of", "it is", "it was",
        "this is", "there is", "there are", "i believe",
        "according to the document", "based on the document",
        "the document states", "the answer is", "in the document",
        "as mentioned", "as stated", "the memo states",
        "the client's name is", "the discount is",
        "the discount applied on ongoing expenses is",
        "the discount applied is",
        "to ensure", "to approve", "to allocate",
        "to track and manage",
        "the purpose of the",
        "is to",
    ]
    for f in fillers:
        s = s.replace(f, "")
    s = re.sub(r",?\s*suggesting.*$", "", s)
    s = re.sub(r",?\s*indicating.*$", "", s)
    s = re.sub(r",?\s*which.*$", "", s)
    s = re.sub(r",?\s*given.*$", "", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def token_f1(pred, ref):
    pt = normalize(pred).split()
    rt = normalize(ref).split()
    if not pt or not rt: return 0.0
    c = sum((Counter(pt) & Counter(rt)).values())
    if c == 0: return 0.0
    return 2 * c / (len(pt) + len(rt))

def exact_match(pred, ref):
    return int(normalize(pred) == normalize(ref))

In [ ]:
#  Run Evaluation

from bert_score import score as bert_score_fn

print(f"\n Run Evaluation on {EVAL_N} samples")
ems, f1s, all_preds, all_refs, all_qs = [], [], [], [], []

for i in range(EVAL_N):
    sample = val_ds[i]
    pred   = run_inference(sample)

    ems.append(exact_match(pred, sample["answer"]))
    f1s.append(token_f1(pred, sample["answer"]))
    all_preds.append(pred)
    all_refs.append(sample["answer"])
    all_qs.append(sample["question"])

    if (i+1) % 10 == 0:
        # compute bertscore for last 10 samples
        P, R, F = bert_score_fn(
            all_preds[i-9:i+1], all_refs[i-9:i+1],
            lang="en",
            model_type="distilbert-base-uncased",
            verbose=False,
            device=DEVICE,
        )
        avg_f1   = np.mean(f1s) * 100
        avg_bert = np.mean(F.tolist()) * 100
        print(f"Evaluated {i+1}/{EVAL_N} | Avg Token F1: {avg_f1:.1f}% | Avg BERTScore: {avg_bert:.1f}%")

# Final BERTScore on all samples
print("\nCompute final BERTScore")
P, R, F = bert_score_fn(
    all_preds, all_refs,
    lang="en",
    model_type="distilbert-base-uncased",
    verbose=False,
    device=DEVICE,
)
bert_f1s    = F.tolist()
bert_prec   = P.tolist()
bert_recall = R.tolist()

# ── Print Results ──────────────────────────────────────────
print(f"\n{'='*55}")
print(f"Final Eval ({EVAL_N} samples) — Checkpoint: epoch_6_valloss_1.149")
print(f"{'='*55}")
print(f"Exact Match     : {np.mean(ems)*100:.1f}%")
print(f"Token F1        : {np.mean(f1s)*100:.1f}%")
print(f"BERTScore F1    : {np.mean(bert_f1s)*100:.1f}%")
print(f"BERTScore P     : {np.mean(bert_prec)*100:.1f}%")
print(f"BERTScore R     : {np.mean(bert_recall)*100:.1f}%")
print(f"{'='*55}")

print(f"\nBERTScore Distribution:")
print(f"  > 0.90 : {sum(1 for x in bert_f1s if x > 0.90)} samples")
print(f"  > 0.80 : {sum(1 for x in bert_f1s if x > 0.80)} samples")
print(f"  > 0.70 : {sum(1 for x in bert_f1s if x > 0.70)} samples")
print(f"  < 0.70 : {sum(1 for x in bert_f1s if x < 0.70)} samples")

# ── Sample Predictions ─────────────────────────────────────
print("\nSample predictions:")
for i in range(min(10, EVAL_N)):
    print(f"\n[{i+1}] Q          : {all_qs[i]}")
    print(f"      Ref        : {all_refs[i]}")
    print(f"      Pred       : {all_preds[i]}")
    print(f"      Token F1   : {f1s[i]:.3f} | BERTScore: {bert_f1s[i]:.3f}")


In [ ]:

# Hallucination Metrics measures how faithful the model's predictions are to the SOURCE CONTENT, not just whether they match the reference answer
# A prediction is "hallucinated" if its faithfulness F1 to content is below threshold AND it doesn't match the reference answer either
# Measures word overlap between prediction and the source content
# Low score = model is generating words not present in the content = hallucination

import re
from collections import Counter
import numpy as np
from bert_score import score as bert_score_fn

# Compute Faithfulness Token F1 

def faithfulness_token_f1(pred, content):
    pt = normalize(pred).split()
    ct = normalize(content).split()
    if not pt or not ct:
        return 0.0
    common = sum((Counter(pt) & Counter(ct)).values())
    if common == 0:
        return 0.0
    precision = common / len(pt)   # how much of the pred is grounded in content
    recall    = common / len(ct)   # how much of content is covered
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# Compute Hallucination Rate


HALLUCINATION_THRESHOLD = 0.15  # tune this based on your domain


def is_hallucinated(pred, content, ref, threshold=HALLUCINATION_THRESHOLD):
    faith_f1 = faithfulness_token_f1(pred, content)
    ref_match = token_f1(pred, ref)
    # hallucinated if not grounded in content AND not matching reference
    return int(faith_f1 < threshold and ref_match < 0.3), faith_f1

# Compute Hallucination Metrics 
print("\nComputing Hallucination Metrics")

faith_f1s        = []
hallucinated     = []
content_bert_f1s = []

all_contents_eval = [val_ds[i]["content"] for i in range(EVAL_N)]

for pred, content, ref in zip(all_preds, all_contents_eval, all_refs):
    f1    = faithfulness_token_f1(pred, content)
    h, _  = is_hallucinated(pred, content, ref)
    faith_f1s.append(f1)
    hallucinated.append(h)

print("Computing BERTScore ")
_, _, F_content = bert_score_fn(
    all_preds,
    all_contents_eval,
    lang="en",
    model_type="distilbert-base-uncased",
    verbose=False,
    device=DEVICE,
)
content_bert_f1s = F_content.tolist()

# Print Metrics
print(f"\n{'='*55}")
print(f"Hallucination Report ({EVAL_N} samples)")
print(f"{'='*55}")
print(f"Faithfulness Token F1  : {np.mean(faith_f1s)*100:.1f}%  (pred vs content)")
print(f"BERTScore F1           : {np.mean(content_bert_f1s)*100:.1f}%  (pred vs content)")
print(f"Hallucination Rate     : {np.mean(hallucinated)*100:.1f}%  (pred grounded in neither content nor ref)")
print(f"{'='*55}")
print(f"\nFaithfulness Distribution (pred vs content):")
print(f"  > 0.50 (well grounded) : {sum(1 for x in faith_f1s if x > 0.50)} samples")
print(f"  0.30–0.50 (partial)    : {sum(1 for x in faith_f1s if 0.30 <= x <= 0.50)} samples")
print(f"  < 0.30 (likely halluc) : {sum(1 for x in faith_f1s if x < 0.30)} samples")
